# Flow Matching

**Domain:** Architectures  ·  **recommended addition**  ·  **runnable:** yes

A refresher on **Flow Matching (FM)** — the simulation-free training objective behind modern
continuous generative models (Stable Diffusion 3, Flux, many speech/video models). If you knew
diffusion and continuous normalizing flows once and want the one-page reload, this is it.

## 1. What & Why

**Flow Matching trains a generative model by regressing a velocity field.** You want to turn an
easy distribution (Gaussian noise) into a hard one (your data). FM does this by defining a
**probability path** `p_t` that morphs noise (`t=0`) into data (`t=1`), and training a neural net
`v_θ(x, t)` to predict the **velocity** that pushes a sample along that path. At sampling time you
start from noise and **integrate an ODE** `dx/dt = v_θ(x, t)` from `t=0` to `t=1`.

**The problem it solves.** Continuous Normalizing Flows (CNFs) are expressive but were historically
trained by *simulating* the ODE during training and backpropagating through the solver — slow and
unstable. Flow Matching gives a **simulation-free** objective: a plain MSE regression with a
closed-form target, no ODE solve in the loop. It's the same "regress a vector field" trick that makes
diffusion practical, but stated directly in velocity/transport terms.

**When to reach for it.** You want high-quality continuous generation (images, audio, video,
molecules) with **stable training** and **flexible, few-step sampling**. With straight-line
("rectified" / optimal-transport) paths you often sample in far fewer steps than a vanilla diffusion
model. It is now the default formulation for new large image/video models.

**When not to.** For exact likelihoods on tabular/low-dim data a classic normalizing flow may be
simpler; for discrete data (text) you want a different formulation; and if you already have a tuned
diffusion pipeline, FM is a re-parameterization, not a magic upgrade.

## 2. Mental Model

**Draw a straight line from a noise point to a data point, and teach the network the arrow along
that line.**

- Pick a real data sample `x₁` and a random noise sample `x₀ ~ N(0, I)`.
- Define the point at time `t` as the **linear interpolation** `xₜ = (1−t)·x₀ + t·x₁`.
- The velocity that carries you along this segment is constant: `x₁ − x₀`. That's the **target**.
- Train `v_θ(xₜ, t)` to predict `x₁ − x₀` with MSE.

Any single `xₜ` lies on *many* such segments (different `x₀`/`x₁` pairs cross there), so the network
can't memorize one arrow — it learns the **average velocity** of all transport paths passing through
that point at that time. That average is exactly the marginal field that transports the whole noise
cloud onto the data manifold. To generate: drop a particle at noise and let this learned wind field
blow it to the data over `t : 0 → 1`.

> Diffusion learns "which way is *less* noisy" (a score). Flow Matching learns "which way to *move*"
> (a velocity). With Gaussian paths the two are linear re-parameterizations of each other.

## 3. Key Concepts

- **Probability path `p_t`** — a time-indexed family of distributions interpolating `p₀ = N(0, I)`
  (noise) and `p₁ ≈ p_data`. FM never needs `p_t` in closed form, only how to *sample* `xₜ`.
- **Velocity / vector field `v(x, t)`** — the instantaneous direction+speed that transports mass
  along the path. Generating = following it; this is the thing the network outputs.
- **Conditional Flow Matching (CFM)** — the key trick. The true *marginal* velocity is intractable,
  but the velocity **conditioned on a single endpoint pair** `(x₀, x₁)` is trivial. Regressing the
  conditional target yields the same gradient as regressing the marginal one — so you train on the
  easy thing and get the right field. This is what makes FM simulation-free.
- **Optimal-Transport / Rectified path** — choosing the *straight* interpolation `xₜ = (1−t)x₀ + tx₁`
  with constant target `x₁ − x₀`. Straight paths → easier-to-integrate ODE → **fewer sampling steps**.
- **ODE sampling** — at inference solve `dx/dt = v_θ(x, t)`. A handful of Euler/Heun/RK steps; no
  stochasticity required (though an equivalent SDE form exists).
- **Time conditioning** — `t` is a genuine input to the network (often via an embedding). Forgetting
  it is the classic bug; the field is *non-stationary*.
- **Relation to diffusion** — Gaussian-path FM, score matching, and `v`-prediction are mutually
  convertible. FM is the transport-centric, often cleaner, statement.

## 4. Setup

The worked examples use only **PyTorch (CPU is fine)**, NumPy, and Matplotlib — a 2-D toy where you
can literally see the noise cloud flow onto the data. No GPU, no downloads. The final cell *optionally*
loads a real pretrained Flow-Matching image model (Stable Diffusion 3) and is gated behind an env var,
so the notebook runs end-to-end without it.

In [1]:
# %pip install numpy torch matplotlib    # CPU build of torch is plenty for the 2-D toy
import math, os
import numpy as np
import torch
import torch.nn as nn

torch.manual_seed(0)
np.random.seed(0)
print("torch", torch.__version__, "| device: cpu (toy is tiny)")

torch 2.12.1 | device: cpu (toy is tiny)


## 5. Worked Examples

### Example 1 — The target distribution and the FM training objective

We'll learn to generate an **8-Gaussians ring** — eight tight blobs on a circle. It's 2-D so we can
plot the whole process. First, the data sampler and a quick look.

In [2]:
def sample_data(n):
    """Eight Gaussian blobs evenly spaced on a circle of radius 4."""
    angles = np.arange(8) * (math.pi / 4)
    centers = np.stack([np.cos(angles), np.sin(angles)], axis=1) * 4.0   # (8, 2)
    idx = np.random.randint(0, 8, size=n)
    pts = centers[idx] + 0.20 * np.random.randn(n, 2)
    return torch.tensor(pts, dtype=torch.float32)

data = sample_data(2000)
print("data shape:", tuple(data.shape),
      "| x-range [%.1f, %.1f]" % (data[:, 0].min(), data[:, 0].max()))
print("8 modes at radius ~", round(float(data.norm(dim=1).mean()), 2))

data shape: (2000, 2) | x-range [-4.6, 4.6]
8 modes at radius ~ 4.01


The velocity network is a tiny MLP taking `(x, t)` → velocity. Nothing fancy: three hidden layers,
SiLU activations. **Note `t` is concatenated as an input** — the field depends on time.

In [3]:
class VelocityField(nn.Module):
    def __init__(self, dim=2, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim + 1, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, dim),
        )

    def forward(self, x, t):           # x: (B, dim), t: (B, 1)
        return self.net(torch.cat([x, t], dim=-1))

model = VelocityField()
n_params = sum(p.numel() for p in model.parameters())
print(f"velocity field: {n_params:,} params")

velocity field: 33,794 params


Now the **whole of Flow Matching** in one training loop. Per step: sample data `x₁`, noise `x₀`,
a random time `t`, form the straight-line point `xₜ`, and regress the constant velocity `x₁ − x₀`.
That MSE *is* the conditional-OT Flow Matching loss.

In [4]:
opt = torch.optim.Adam(model.parameters(), lr=2e-3)

for step in range(3000):
    x1 = sample_data(512)                 # data endpoint
    x0 = torch.randn_like(x1)             # noise endpoint  ~ N(0, I)
    t  = torch.rand(x1.size(0), 1)        # time ~ U(0, 1)
    xt = (1 - t) * x0 + t * x1            # straight-line interpolant
    target = x1 - x0                      # constant conditional velocity

    v = model(xt, t)
    loss = ((v - target) ** 2).mean()     # simulation-free regression

    opt.zero_grad(); loss.backward(); opt.step()
    if step % 500 == 0 or step == 2999:
        print(f"step {step:4d}  loss {loss.item():.4f}")

step    0  loss 9.1452


step  500  loss 3.8039


step 1000  loss 3.3261


step 1500  loss 3.7267


step 2000  loss 3.4019


step 2500  loss 3.5658


step 2999  loss 3.3935


### Example 2 — Sampling: integrate the ODE from noise to data

Generation drops particles at noise (`t=0`) and pushes them with the learned velocity to `t=1` using
plain **Euler** steps. We sweep the step count to show that **straight paths sample in very few
steps** — a defining selling point of rectified Flow Matching.

In [5]:
@torch.no_grad()
def generate(model, n, steps):
    x = torch.randn(n, 2)                 # start at noise, t = 0
    dt = 1.0 / steps
    for i in range(steps):
        t = torch.full((n, 1), i * dt)
        x = x + model(x, t) * dt          # Euler: x <- x + v*dt
    return x

# Coverage metric: a sample "hits" a mode if it lands within 1.0 of a center.
angles = np.arange(8) * (math.pi / 4)
centers = torch.tensor(np.stack([np.cos(angles), np.sin(angles)], 1) * 4.0,
                       dtype=torch.float32)

def coverage(samples):
    d = torch.cdist(samples, centers)            # (n, 8)
    nearest = d.min(dim=1)
    hit = nearest.values < 1.0
    modes_found = len(torch.unique(nearest.indices[hit]))
    return hit.float().mean().item(), modes_found

for steps in (1, 2, 5, 20, 100):
    s = generate(model, 4000, steps)
    frac, modes = coverage(s)
    print(f"{steps:4d} Euler steps -> {frac*100:5.1f}% near a mode, {modes}/8 modes covered")

   1 Euler steps ->   0.0% near a mode, 0/8 modes covered
   2 Euler steps ->  39.6% near a mode, 8/8 modes covered
   5 Euler steps ->  91.5% near a mode, 8/8 modes covered
  20 Euler steps ->  98.3% near a mode, 8/8 modes covered
 100 Euler steps ->  99.3% near a mode, 8/8 modes covered


A single step collapses toward the data *mean* (the field at `t=0` averages over all data), but
just a handful of steps recovers all eight modes — what makes FM / rectified-flow samplers cheap.
Now visualize: real data, generated samples, and a few **particle trajectories** flowing out from
noise.

In [6]:
import matplotlib
matplotlib.use("Agg")          # headless-safe; remove for interactive plots
import matplotlib.pyplot as plt

@torch.no_grad()
def trajectories(model, n, steps=60):
    x = torch.randn(n, 2)
    path = [x.clone()]
    dt = 1.0 / steps
    for i in range(steps):
        t = torch.full((n, 1), i * dt)
        x = x + model(x, t) * dt
        path.append(x.clone())
    return torch.stack(path)               # (steps+1, n, 2)

gen = generate(model, 2000, steps=50)
traj = trajectories(model, 12)

fig, ax = plt.subplots(1, 3, figsize=(13, 4.2))
ax[0].scatter(data[:, 0], data[:, 1], s=4, alpha=0.4); ax[0].set_title("Real data (8 Gaussians)")
ax[1].scatter(gen[:, 0],  gen[:, 1],  s=4, alpha=0.4, color="C1"); ax[1].set_title("FM samples (50 steps)")
ax[2].scatter(data[:, 0], data[:, 1], s=3, alpha=0.15)
for k in range(traj.shape[1]):
    ax[2].plot(traj[:, k, 0], traj[:, k, 1], lw=1.0, alpha=0.8)
ax[2].scatter(traj[0, :, 0], traj[0, :, 1], color="k", s=15, zorder=3)   # noise starts
ax[2].set_title("Particle trajectories: noise -> data")
for a in ax: a.set_xlim(-6, 6); a.set_ylim(-6, 6); a.set_aspect("equal")
plt.tight_layout(); plt.savefig("flow_matching_demo.png", dpi=90)
print("saved flow_matching_demo.png")
plt.show()

saved flow_matching_demo.png


/var/folders/p8/sm5jmh055md_zzhhn1mfgyw80000gn/T/ipykernel_70303/4026483268.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Example 3 — FM velocity ⇄ diffusion score are the same field, re-parameterized

A common point of confusion: "is this diffusion or not?" For the Gaussian path used above, the FM
**velocity** and the diffusion **score** `∇ₓ log pₜ(x)` are linked by a closed-form, time-dependent
map. We verify it analytically on the straight path with a single data point, where everything is in
closed form: compute the velocity two ways — directly, and *converted from the score* — and check
they agree.

In [7]:
# Straight path x_t = (1-t)*x0 + t*x1, with x0 ~ N(0,1) and a single data point x1.
# Marginal:  x_t ~ N(mu_t, s_t^2),  mu_t = t*x1,  s_t = 1 - t.
#   velocity (direct):  v(x,t) = (x1 - x) / (1 - t)        # since x0 = (x - t*x1)/(1-t)
#   score:              g(x,t) = -(x - mu_t) / s_t**2
# Conversion derived from the path:  v(x,t) = (x + (1 - t)*g(x,t)) / t
x1 = 3.0
print(f"{'t':>4} {'v_direct':>10} {'v_from_score':>13} {'match':>7}")
for t in (0.2, 0.5, 0.8):
    x = 1.0                                         # arbitrary query point
    mu, s = t * x1, (1 - t)
    v_direct     = (x1 - x) / (1 - t)
    g            = -(x - mu) / s**2                 # score of N(x; mu, s^2)
    v_from_score = (x + (1 - t) * g) / t
    print(f"{t:>4.1f} {v_direct:>10.4f} {v_from_score:>13.4f} "
          f"{str(bool(np.isclose(v_direct, v_from_score))):>7}")

   t   v_direct  v_from_score   match
 0.2     2.5000        2.5000    True
 0.5     4.0000        4.0000    True
 0.8    10.0000       10.0000    True


The two columns match at every `t`: velocity and score are the *same* field in different
coordinates (given the schedule `aₜ = 1−t, bₜ = t`). Practically this is why a Flow-Matching
checkpoint and a `v`/score-diffusion checkpoint are inter-convertible, and why you can run FM models
with diffusion ODE samplers and vice-versa.

In [8]:
# OPTIONAL: a real pretrained Flow-Matching image model (Stable Diffusion 3 uses FM / rectified flow).
# Downloads several GB of weights and needs a HF token, so it is gated. Set ALLOW_FM_DOWNLOAD=1
# (and accept the model license / `huggingface-cli login`) to actually run it.
if os.getenv("ALLOW_FM_DOWNLOAD") == "1":
    from diffusers import StableDiffusion3Pipeline      # pip install diffusers transformers accelerate
    pipe = StableDiffusion3Pipeline.from_pretrained(
        "stabilityai/stable-diffusion-3-medium-diffusers", torch_dtype=torch.float16).to("cuda")
    image = pipe("a corgi astronaut, studio photo", num_inference_steps=28).images[0]
    image.save("sd3_sample.png")
    print("saved sd3_sample.png")
else:
    print("Skipped SD3 download (set ALLOW_FM_DOWNLOAD=1 to run).")
    print("Same idea, scaled up: SD3/Flux regress a velocity field over a rectified path in")
    print("VAE-latent space, conditioned on text, and sample by integrating the ODE in ~20-30 steps.")

Skipped SD3 download (set ALLOW_FM_DOWNLOAD=1 to run).
Same idea, scaled up: SD3/Flux regress a velocity field over a rectified path in
VAE-latent space, conditioned on text, and sample by integrating the ODE in ~20-30 steps.


## 6. Gotchas & Pitfalls

- **Forgetting the time input.** `v_θ` *must* take `t`. The field is non-stationary; a network that
  ignores `t` learns a blurry time-average and samples poorly. (Real models embed `t`, e.g. sinusoidal
  / Fourier features, instead of raw concatenation.)
- **Sampling-step / discretization error.** Euler with too few steps overshoots curved paths. Straight
  (OT/rectified) paths keep trajectories near-linear so few steps suffice; non-straight paths need more
  steps or a higher-order solver (Heun/RK). Always sweep step count.
- **Conditional vs marginal confusion.** You regress the *conditional* velocity `x₁ − x₀` for sampled
  pairs, but the network learns the *marginal* field because many pairs cross each point. Don't expect
  any single prediction to equal a particular `x₁ − x₀`.
- **Schedule / endpoint mismatch.** Train and sample must agree on the path: same `aₜ, bₜ`, same
  `t=0`→noise / `t=1`→data orientation. Flipping the time direction silently produces noise.
- **`σ_min` / exact-noise paths.** The pure linear path sends variance to 0 at `t=1`; some
  formulations add a small `σ_min` so the conditional path stays a proper Gaussian. Minor, but it
  changes the exact target.
- **Loss looks low but samples are bad.** MSE on velocity is only a proxy. Judge with held-out sample
  quality (coverage, FID), not the training loss alone.
- **Coupling matters.** Independent `(x₀, x₁)` pairing is the baseline; minibatch-OT or rectified-flow
  *reflow* re-pairs endpoints to straighten paths further and cut sampling steps.

## 7. When to Use vs Alternatives

| Approach | Training | Sampling | Likelihood | Notes |
|---|---|---|---|---|
| **Flow Matching (rectified/OT)** | simulation-free MSE on velocity; very stable | ODE, **few steps** | via ODE (instantaneous change-of-vars) | SOTA for new image/video/audio models (SD3, Flux) |
| **Diffusion (DDPM/score)** | denoising / score MSE; stable | ODE/SDE, often many steps (distillation helps) | yes | mature ecosystem; FM is a cleaner re-statement |
| **Continuous Normalizing Flows (CNF)** | *simulate* ODE in the loop — slow | ODE | exact-ish | FM is the simulation-free way to train these |
| **Discrete Normalizing Flows** | exact MLE, invertible blocks | one forward pass | **exact, fast** | best when you need exact density on low-dim data |
| **GANs** | adversarial min-max; can be unstable | **one** forward pass | none | fastest sampling; no density; mode-collapse risk |

**Rules of thumb.** Building a new high-quality continuous generator → **Flow Matching with rectified
paths**: stable training, cheap sampling, scales. Need **exact likelihoods** on modest-dimensional
data → a classic normalizing flow. Need **single-step** generation and don't care about density →
GAN (or distill an FM/diffusion model). Already invested in a diffusion stack → FM is a
re-parameterization you can adopt incrementally, not a separate universe.

See also the sibling notebooks: [`diffusion-models`](diffusion-models.ipynb),
[`normalizing-flows`](normalizing-flows.ipynb), and [`diffusion-transformer`](diffusion-transformer.ipynb)
(the backbone SD3/Flux pair with FM).

## 8. Resources

- **Flow Matching for Generative Modeling** — Lipman, Chen, Ben-Hamu, Nickel, Le (2022), the founding
  paper: https://arxiv.org/abs/2210.02747
- **Flow Matching Guide and Code** — Lipman et al. (2024), a thorough tutorial + reference library:
  https://arxiv.org/abs/2412.06264  ·  code: https://github.com/facebookresearch/flow_matching
- **Rectified Flow** — Liu, Gong, Liu (2022), straight paths and *reflow* for few-step sampling:
  https://arxiv.org/abs/2209.03003
- **Stochastic Interpolants** — Albergo, Vanden-Eijnden et al., the unifying noise↔data interpolation
  view: https://arxiv.org/abs/2303.08797
- **Scaling Rectified Flow Transformers (SD3)** — Esser et al. (2024), FM at production scale:
  https://arxiv.org/abs/2403.03206
- **An Introduction to Flow Matching** — Cambridge MLG blog, clear visual walkthrough:
  https://mlg.eng.cam.ac.uk/blog/2024/01/20/flow-matching.html